# Semi-Parametric Setup: End-to-End Sample Generation

This notebook generates a complete event sample with:
- Events at the SM benchmark and all morphing basis points
- Systematic (scale) variation weights per event (7-point grid: mu_R, mu_F, correlated)
- Truth matrix element weights at all benchmarks (usable for morphing to arbitrary theta)
- A standalone export file for downstream use outside MadMiner

**Batch mode:** Set `BATCH_ID` below and re-run to generate independent batches.
Use `aggregate_and_convert.py` to combine them afterwards.

## 0. Environment Setup

Install the local (patched) MadMiner and set `LD_LIBRARY_PATH` so MadGraph can access LHAPDF for scale systematics. **Restart the kernel after running the pip install cell if this is your first time.**

In [1]:
import subprocess
subprocess.check_call(["pip", "install", "-e", "/home/shared/madminer"])

Obtaining file:///home/shared/madminer
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for madminer (pyproject.toml): started
  Building editable for madminer (pyproject.toml): finished with status 'done'
  Created wheel for madminer: filename=madminer-0.9.7.dev25-py3-none-any.whl size=5682 sha256=b7d2036827307a14b0f4a85afd9b3feaf08868ba50c8769c3d8bff7806980b61
  Stored in directory: /tmp/pip-ephem-wheel-cache-_147mwd6/whe


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


0

In [2]:
import os

os.environ["LD_LIBRARY_PATH"] = (
    "/madminer/software/MG5_aMC_v2_9_16/HEPTools/lhapdf6_py3/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)

In [3]:
# ============================================================
# BATCH CONFIGURATION - increment this for each independent run
# ============================================================
BATCH_ID = 1

import logging
import numpy as np
import h5py

from madminer.core import MadMiner
from madminer.lhe import LHEReader
from madminer.analysis import DataAnalyzer
from madminer.sampling import combine_and_shuffle
from particle import Particle

mg_dir = os.getenv("MG_FOLDER_PATH")

# Logging
logging.basicConfig(
    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",
    datefmt="%H:%M",
    level=logging.INFO,
)
for key in logging.Logger.manager.loggerDict:
    if "madminer" not in key:
        logging.getLogger(key).setLevel(logging.WARNING)

print(f"Batch ID: {BATCH_ID}")

Batch ID: 1


## 1. Parameter Space, Benchmarks, and Morphing

Define the EFT parameter space (two Wilson coefficients), manual benchmarks, and let MadMiner optimize the morphing basis.

In [4]:
miner = MadMiner()

miner.add_parameter(
    lha_block="dim6",
    lha_id=2,
    parameter_name="CWL2",
    morphing_max_power=2,
    param_card_transform="16.52*theta",
    parameter_range=(-20.0, 20.0),
)
miner.add_parameter(
    lha_block="dim6",
    lha_id=5,
    parameter_name="CPWL2",
    morphing_max_power=2,
    param_card_transform="16.52*theta",
    parameter_range=(-20.0, 20.0),
)

# Benchmarks
miner.add_benchmark({"CWL2": 0.0, "CPWL2": 0.0}, "sm")
miner.add_benchmark({"CWL2": 15.2, "CPWL2": 0.1}, "w")
miner.add_benchmark({"CWL2": -15.4, "CPWL2": 0.2}, "neg_w")
miner.add_benchmark({"CWL2": 0.3, "CPWL2": 15.1}, "ww")
miner.add_benchmark({"CWL2": 0.4, "CPWL2": -15.3}, "neg_ww")

# Morphing (will add 1 extra basis point to complete the basis)
miner.set_morphing(include_existing_benchmarks=True, max_overall_power=2)

20:41 madminer.core.madmin INFO    Adding parameter: CWL2 (LHA: dim6 2, Power: 2, Range: (-20.0, 20.0))
20:41 madminer.core.madmin WARNING Resetting benchmarks and morphing
20:41 madminer.core.madmin INFO    Adding parameter: CPWL2 (LHA: dim6 5, Power: 2, Range: (-20.0, 20.0))
20:41 madminer.core.madmin WARNING Resetting benchmarks and morphing
20:41 madminer.core.madmin INFO    Added benchmark sm: CWL2 = 0.00e+00, CPWL2 = 0.00e+00
20:41 madminer.core.madmin INFO    Added benchmark w: CWL2 = 15.20, CPWL2 = 0.10
20:41 madminer.core.madmin INFO    Added benchmark neg_w: CWL2 = -1.54e+01, CPWL2 = 0.20
20:41 madminer.core.madmin INFO    Added benchmark ww: CWL2 = 0.30, CPWL2 = 15.10
20:41 madminer.core.madmin INFO    Added benchmark neg_ww: CWL2 = 0.40, CPWL2 = -1.53e+01
20:41 madminer.core.madmin INFO    Optimizing basis for morphing
20:41 madminer.core.madmin INFO    Set up morphing with 2 parameters, 6 morphing components, 5 predefined basis points, and 1 new basis points


## 2. Add Systematics and Save Setup

Add scale variation systematics as 3 separate nuisance parameters covering the 7-point grid:

| Systematic | Varies | Extracted points |
|-----------|--------|------------------|
| `scale_mur` | mu_R only (mu_F=1) | (0.5, 1.0) and (2.0, 1.0) |
| `scale_muf` | mu_F only (mu_R=1) | (1.0, 0.5) and (1.0, 2.0) |
| `scale_corr` | both together | (0.5, 0.5) and (2.0, 2.0) |

This gives 6 off-nominal + 1 nominal = 7 grid points, excluding anti-correlated extremes.

In [5]:
miner.add_systematics(effect="scale", systematic_name="scale_mur", scale="mur")
miner.add_systematics(effect="scale", systematic_name="scale_muf", scale="muf")
miner.add_systematics(effect="scale", systematic_name="scale_corr", scale="mu")

miner.save("data/setup_semi_parametric.h5")

20:41 madminer.core.madmin INFO    Saving setup (including morphing) to data/setup_semi_parametric.h5


## 3. Event Generation with MadGraph

Generate signal events at the SM benchmark and at each additional benchmark.
All systematics are requested for each run so MadGraph computes the full scale variation grid.

**Note:** Change `run_card_file` to `cards/run_card_signal_large.dat` (50k events) for meaningful statistics.

In [6]:
all_systematics = ["scale_mur", "scale_muf", "scale_corr"]

# Only pass scale_corr (scale="mu") to MadGraph — it computes the full 3x3
# mu_R x mu_F grid regardless. The other systematics are extracted during
# LHE parsing from the same weight groups.
mg_systematics = ["scale_corr"]

# Signal: sample at SM benchmark
miner.run(
    sample_benchmark="sm",
    mg_directory=mg_dir,
    mg_process_directory=f"./mg_processes/signal_semi_sm_b{BATCH_ID}",
    proc_card_file="cards/proc_card_signal.dat",
    param_card_template_file="cards/param_card_template.dat",
    run_card_file="cards/run_card_signal_large.dat",
    log_directory=f"logs/signal_semi_b{BATCH_ID}",
    systematics=mg_systematics,
)

20:41 madminer.utils.inter INFO    Generating MadGraph process folder from cards/proc_card_signal.dat at ./mg_processes/signal_semi_sm_b1
20:41 madminer.utils.inter INFO    Calling MadGraph: /madminer/software/MG5_aMC_v2_9_16/bin/mg5_aMC /tmp/generate.mg5
20:41 madminer.core.madmin INFO    Run 0
20:41 madminer.core.madmin INFO      Sampling from benchmark: sm
20:41 madminer.core.madmin INFO      Original run card:       cards/run_card_signal_large.dat
20:41 madminer.core.madmin INFO      Original Pythia8 card:   None
20:41 madminer.core.madmin INFO      Original config card:    None
20:41 madminer.core.madmin INFO      Copied run card:         madminer/cards/run_card_0.dat
20:41 madminer.core.madmin INFO      Copied Pythia8 card:     None
20:41 madminer.core.madmin INFO      Copied config card:      None
20:41 madminer.core.madmin INFO      Param card:              madminer/cards/param_card_0.dat
20:41 madminer.core.madmin INFO      Reweight card:           madminer/cards/reweight_card

In [7]:
# Signal: sample at additional benchmarks for better phase space coverage.
additional_benchmarks = ["w", "ww", "neg_w", "neg_ww"]

for bm in additional_benchmarks:
    miner.run(
        sample_benchmark=bm,
        mg_directory=mg_dir,
        mg_process_directory=f"./mg_processes/signal_semi_{bm}_b{BATCH_ID}",
        proc_card_file="cards/proc_card_signal.dat",
        param_card_template_file="cards/param_card_template.dat",
        run_card_file="cards/run_card_signal_large.dat",
        log_directory=f"logs/signal_semi_b{BATCH_ID}",
        systematics=mg_systematics,
    )

21:12 madminer.utils.inter INFO    Generating MadGraph process folder from cards/proc_card_signal.dat at ./mg_processes/signal_semi_w_b1
21:12 madminer.utils.inter INFO    Calling MadGraph: /madminer/software/MG5_aMC_v2_9_16/bin/mg5_aMC /tmp/generate.mg5
21:12 madminer.core.madmin INFO    Run 0
21:12 madminer.core.madmin INFO      Sampling from benchmark: w
21:12 madminer.core.madmin INFO      Original run card:       cards/run_card_signal_large.dat
21:12 madminer.core.madmin INFO      Original Pythia8 card:   None
21:12 madminer.core.madmin INFO      Original config card:    None
21:12 madminer.core.madmin INFO      Copied run card:         madminer/cards/run_card_0.dat
21:12 madminer.core.madmin INFO      Copied Pythia8 card:     None
21:12 madminer.core.madmin INFO      Copied config card:      None
21:12 madminer.core.madmin INFO      Param card:              madminer/cards/param_card_0.dat
21:12 madminer.core.madmin INFO      Reweight card:           madminer/cards/reweight_card_0

In [8]:
# Background generation moved to semi_parametric_background.ipynb
# Run it separately — it takes much longer than signal.

## 4. Parse LHE Files, Define Observables, and Run Analysis

Read the generated LHE files, define detector smearing, observables, and selection cuts, then extract all event data.

In [9]:
lhe = LHEReader("data/setup_semi_parametric.h5")

# Signal sample from SM benchmark
lhe.add_sample(
    lhe_filename=f"mg_processes/signal_semi_sm_b{BATCH_ID}/Events/run_01/unweighted_events.lhe.gz",
    sampled_from_benchmark="sm",
    is_background=False,
    k_factor=1.1,
    systematics=all_systematics,
)

# Signal samples from additional benchmarks
for bm in additional_benchmarks:
    lhe.add_sample(
        lhe_filename=f"mg_processes/signal_semi_{bm}_b{BATCH_ID}/Events/run_01/unweighted_events.lhe.gz",
        sampled_from_benchmark=bm,
        is_background=False,
        k_factor=1.1,
        systematics=all_systematics,
    )

# To include background, run semi_parametric_background.ipynb first, then uncomment:
# lhe.add_sample(
#     lhe_filename=f"mg_processes/bkg_semi_b{BATCH_ID}/Events/run_01/unweighted_events.lhe.gz",
#     sampled_from_benchmark="sm",
#     is_background=True,
#     k_factor=1.1,
#     systematics=all_systematics,
# )

23:25 madminer.utils.inter INFO    HDF5 file does not contain nuisance parameters information
23:25 madminer.utils.inter INFO    HDF5 file does not contain finite difference information
23:25 madminer.utils.inter INFO    HDF5 file does not contain observables information
23:25 madminer.utils.inter INFO    HDF5 file does not contain sample summary information
23:25 madminer.utils.inter INFO    HDF5 file does not contain sample information


In [10]:
# Detector smearing for jet-like partons
particles = [
    *Particle.findall(lambda p: p.pdgid.is_quark),
    *Particle.findall(pdg_name="g"),
]

lhe.set_smearing(
    pdgids=[int(p.pdgid) for p in particles],
    energy_resolution_abs=0.0,
    energy_resolution_rel=0.1,
    pt_resolution_abs=None,
    pt_resolution_rel=None,
    eta_resolution_abs=0.1,
    eta_resolution_rel=0.0,
    phi_resolution_abs=0.1,
    phi_resolution_rel=0.0,
)

# Observables
lhe.add_observable("pt_j1", "j[0].pt", required=False, default=0.0)
lhe.add_observable(
    "delta_phi_jj",
    "j[0].deltaphi(j[1]) * (-1.0 + 2.0 * float(j[0].eta > j[1].eta))",
    required=True,
)
lhe.add_observable("met", "met.pt", required=True)

# Selection cuts
lhe.add_cut("(a[0] + a[1]).m > 122.0")
lhe.add_cut("(a[0] + a[1]).m < 128.0")
lhe.add_cut("pt_j1 > 30.0")

In [11]:
lhe.analyse_samples()
lhe.save(f"data/lhe_data_semi_parametric_b{BATCH_ID}.h5")

23:25 madminer.lhe.lhe_rea INFO    Analysing LHE sample mg_processes/signal_semi_sm_b1/Events/run_01/unweighted_events.lhe.gz: Calculating 3 observables, requiring 3 selection cuts, using 0 efficiency factors, associated with systematicsscale_mur, scale_muf, scale_corr
00:10 madminer.utils.inter INFO      50000 / 50000 events pass cut Cut(name='CUT', val_expression='(a[0] + a[1]).m > 122.0', is_required=False)
00:10 madminer.utils.inter INFO      50000 / 50000 events pass cut Cut(name='CUT', val_expression='(a[0] + a[1]).m < 128.0', is_required=False)
00:10 madminer.utils.inter INFO      49272 / 50000 events pass cut Cut(name='CUT', val_expression='pt_j1 > 30.0', is_required=False)
00:10 madminer.utils.inter INFO      49272 events pass all cuts/efficiencies
00:10 madminer.lhe.lhe_rea INFO    Analysing LHE sample mg_processes/signal_semi_w_b1/Events/run_01/unweighted_events.lhe.gz: Calculating 3 observables, requiring 3 selection cuts, using 0 efficiency factors, associated with systema

## 5. Verify the MadMiner HDF5 File

Quick sanity check: load the file back and inspect what's inside.

In [12]:
da = DataAnalyzer(f"data/lhe_data_semi_parametric_b{BATCH_ID}.h5", include_nuisance_parameters=True)

# Physical (non-nuisance) benchmark names
all_bm_names = list(da.benchmarks.keys())
bm_nuisance_flags = da.benchmark_nuisance_flags
benchmark_names_phys = [n for n, is_nuis in zip(all_bm_names, bm_nuisance_flags) if not is_nuis]

print(f"Batch {BATCH_ID}")
print(f"Parameters:          {list(da.parameters.keys())}")
print(f"Benchmarks (phys):   {benchmark_names_phys}")
print(f"Benchmarks (all):    {all_bm_names}")
print(f"Observables:         {list(da.observables.keys())}")
print(f"Systematics:         {list(da.systematics.keys())}")
print(f"Nuisance parameters: {list(da.nuisance_parameters.keys())}")
print(f"Morphing available:  {da.morpher is not None}")
print(f"Nuisance morpher:    {da.nuisance_morpher is not None}")

03:08 madminer.analysis.da INFO    Loading data from data/lhe_data_semi_parametric_b1.h5
03:08 madminer.utils.inter INFO    HDF5 file does not contain finite difference information
03:08 madminer.analysis.da INFO    Found 2 parameters
03:08 madminer.analysis.da INFO      0: CWL2 (LHA: dim6 2, Power: 2, Range: (-20.0, 20.0))
03:08 madminer.analysis.da INFO      1: CPWL2 (LHA: dim6 5, Power: 2, Range: (-20.0, 20.0))
03:08 madminer.analysis.da INFO    Found 3 nuisance parameters
03:08 madminer.analysis.da INFO      0: scale_mur_nuisance_param_0 (Systematic: scale_mur, Benchmarks: scale_mur_nuisance_param_0_benchmark_0 | scale_mur_nuisance_param_0_benchmark_1)
03:08 madminer.analysis.da INFO      1: scale_muf_nuisance_param_0 (Systematic: scale_muf, Benchmarks: scale_muf_nuisance_param_0_benchmark_0 | scale_muf_nuisance_param_0_benchmark_1)
03:08 madminer.analysis.da INFO      2: scale_corr_nuisance_param_0 (Systematic: scale_corr, Benchmarks: scale_corr_nuisance_param_0_benchmark_0 | scal

Batch 1
Parameters:          ['CWL2', 'CPWL2']
Benchmarks (phys):   ['sm', 'w', 'neg_w', 'ww', 'neg_ww', 'morphing_basis_vector_5']
Benchmarks (all):    ['sm', 'w', 'neg_w', 'ww', 'neg_ww', 'morphing_basis_vector_5', 'scale_corr_nuisance_param_0_benchmark_0', 'scale_corr_nuisance_param_0_benchmark_1', 'scale_muf_nuisance_param_0_benchmark_0', 'scale_muf_nuisance_param_0_benchmark_1', 'scale_mur_nuisance_param_0_benchmark_0', 'scale_mur_nuisance_param_0_benchmark_1']
Observables:         ['pt_j1', 'delta_phi_jj', 'met']
Systematics:         ['scale_mur', 'scale_muf', 'scale_corr']
Nuisance parameters: ['scale_mur_nuisance_param_0', 'scale_muf_nuisance_param_0', 'scale_corr_nuisance_param_0']
Morphing available:  True
Nuisance morpher:    True


In [13]:
# Load all events with benchmark weights
x_all, weights_all = da.weighted_events(theta=None)

print(f"Observations shape:  {x_all.shape}  (n_events, n_observables)")
print(f"Weights shape:       {weights_all.shape}  (n_events, n_benchmarks)")
print(f"\nFirst event observables: {x_all[0]}")
print(f"First event weights:     {weights_all[0]}")

Observations shape:  (249144, 3)  (n_events, n_observables)
Weights shape:       (249144, 12)  (n_events, n_benchmarks)

First event observables: [1390.94396711   -1.54012426  200.65773576]
First event weights:     [1.21405780e-12 1.84235351e-07 1.92041296e-07 2.46121128e-09
 4.61999082e-09 2.48183664e-07 1.05219691e-12 1.41507996e-12
 1.05219691e-12 1.41507996e-12 1.21405780e-12 1.21405780e-12]


## 6. Export Standalone File

Extract everything from the MadMiner HDF5 and save a self-contained file with:

| Dataset | Shape | Description |
|---------|-------|-------------|
| `observables` | (n_events, n_obs) | Kinematic observables per event |
| `weights_benchmarks` | (n_events, n_benchmarks_phys) | ME weights at each physical benchmark |
| `weights_nuisance_up` | (n_events,) | Weights at nuisance +1sigma (scale up) |
| `weights_nuisance_down` | (n_events,) | Weights at nuisance -1sigma (scale down) |
| `nuisance_a` | (n_nuisance, n_events) | Linear nuisance morpher coefficients |
| `nuisance_b` | (n_nuisance, n_events) | Quadratic nuisance morpher coefficients |
| `sampling_ids` | (n_events,) | Which benchmark each event was generated at (-1 = background) |
| `morphing_matrix` | (n_benchmarks, n_components) | Morphing matrix for reweighting to arbitrary theta |
| `benchmark_names` | (n_benchmarks_phys,) | Benchmark names |
| `benchmark_values` | (n_benchmarks_phys, n_params) | Parameter values at each benchmark |
| `parameter_names` | (n_params,) | Parameter names |
| `observable_names` | (n_obs,) | Observable names |

With `morphing_matrix` and `benchmark_values`, you can compute `w(x|theta)` for any theta outside MadMiner.

In [14]:
output_file = f"data/semi_parametric_samples_batch{BATCH_ID}.h5"

# -- Collect metadata --
n_phys = len(benchmark_names_phys)
param_names = list(da.parameters.keys())
obs_names = list(da.observables.keys())

# Benchmark parameter values (n_benchmarks_phys, n_params)
benchmark_values = np.array([
    [da.benchmarks[bm].values[p] for p in param_names]
    for bm in benchmark_names_phys
])

# -- Event-level data --
x_all, w_all = da.weighted_events(theta=None)
w_phys = w_all[:, :n_phys]

# Sampling benchmark IDs
with h5py.File(f"data/lhe_data_semi_parametric_b{BATCH_ID}.h5", "r") as f:
    sampling_ids = f["samples/sampling_benchmarks"][()]

# -- Nuisance information --
nuisance_cols = {}
for npar_name, npar_obj in da.nuisance_parameters.items():
    if npar_obj.benchmark_pos and npar_obj.benchmark_pos in all_bm_names:
        nuisance_cols[f"{npar_name}_up"] = all_bm_names.index(npar_obj.benchmark_pos)
    if npar_obj.benchmark_neg and npar_obj.benchmark_neg in all_bm_names:
        nuisance_cols[f"{npar_name}_down"] = all_bm_names.index(npar_obj.benchmark_neg)

# Extract all nuisance weight columns
w_nuisance = {}
for col_name, col_idx in nuisance_cols.items():
    w_nuisance[col_name] = w_all[:, col_idx]

# Nuisance morpher coefficients
if da.nuisance_morpher is not None:
    nuisance_a = da.nuisance_morpher.calculate_a(w_all)
    nuisance_b = da.nuisance_morpher.calculate_b(w_all)
else:
    nuisance_a = np.zeros((0, x_all.shape[0]))
    nuisance_b = np.zeros((0, x_all.shape[0]))

# -- Morphing matrix --
morphing_matrix = da.morpher.morphing_matrix if da.morpher is not None else None
morphing_components = da.morpher.components if da.morpher is not None else None

print(f"Batch {BATCH_ID}")
print(f"Events:              {x_all.shape[0]}")
print(f"Observables:         {x_all.shape[1]} {obs_names}")
print(f"Physical benchmarks: {n_phys} {benchmark_names_phys}")
print(f"Nuisance columns:    {list(nuisance_cols.keys())}")
print(f"Nuisance a shape:    {nuisance_a.shape}")
print(f"Morphing matrix:     {morphing_matrix.shape if morphing_matrix is not None else 'N/A'}")

Batch 1
Events:              249144
Observables:         3 ['pt_j1', 'delta_phi_jj', 'met']
Physical benchmarks: 6 ['sm', 'w', 'neg_w', 'ww', 'neg_ww', 'morphing_basis_vector_5']
Nuisance columns:    ['scale_mur_nuisance_param_0_up', 'scale_mur_nuisance_param_0_down', 'scale_muf_nuisance_param_0_up', 'scale_muf_nuisance_param_0_down', 'scale_corr_nuisance_param_0_up', 'scale_corr_nuisance_param_0_down']
Nuisance a shape:    (3, 249144)
Morphing matrix:     (6, 6)


In [15]:
# -- Write standalone HDF5 --
with h5py.File(output_file, "w") as f:
    # Event data
    f.create_dataset("observables", data=x_all)
    f.create_dataset("weights_benchmarks", data=w_phys)
    f.create_dataset("sampling_ids", data=sampling_ids)

    # Nuisance weights (one dataset per variation)
    nuis_grp = f.create_group("nuisance_weights")
    for col_name, col_data in w_nuisance.items():
        nuis_grp.create_dataset(col_name, data=col_data)

    # Nuisance morpher coefficients
    f.create_dataset("nuisance_a", data=nuisance_a)
    f.create_dataset("nuisance_b", data=nuisance_b)

    # Morphing
    if morphing_matrix is not None:
        f.create_dataset("morphing_matrix", data=morphing_matrix)
    if morphing_components is not None:
        f.create_dataset("morphing_components", data=morphing_components)

    # Metadata
    f.create_dataset("benchmark_names", data=np.array(benchmark_names_phys, dtype="S256"))
    f.create_dataset("benchmark_values", data=benchmark_values)
    f.create_dataset("parameter_names", data=np.array(param_names, dtype="S256"))
    f.create_dataset("observable_names", data=np.array(obs_names, dtype="S256"))
    f.create_dataset("nuisance_parameter_names", data=np.array(list(da.nuisance_parameters.keys()), dtype="S256"))
    f.create_dataset("systematics_names", data=np.array(list(da.systematics.keys()), dtype="S256"))
    f.attrs["batch_id"] = BATCH_ID

print(f"Saved batch {BATCH_ID} to {output_file}")

Saved batch 1 to data/semi_parametric_samples_batch1.h5


## 7. Verify: Reconstruct Weights at Arbitrary Theta

Demonstrate that the exported file is self-contained: load it back without MadMiner and compute `w(x|theta)` at an arbitrary parameter point using the morphing matrix.

In [16]:
# Load the standalone file and verify
with h5py.File(output_file, "r") as f:
    print(f"Batch {f.attrs['batch_id']}")
    print(f"Events: {f['observables'].shape[0]}")
    print(f"Observables: {f['observables'].shape[1]}")
    print(f"Benchmark weights: {f['weights_benchmarks'].shape}")
    print(f"Nuisance weight datasets: {list(f['nuisance_weights'].keys())}")
    print(f"Nuisance a/b shape: {f['nuisance_a'].shape}")
    print(f"Morphing matrix: {f['morphing_matrix'].shape}")
    print()

    # Quick validation: check all weights are nonzero
    w = f['weights_benchmarks'][()]
    bm_names = [s.decode() for s in f['benchmark_names'][()]]
    for i, name in enumerate(bm_names):
        col = w[:, i]
        print(f"  {name:30s}  mean={col.mean():.4e}  nonzero={np.count_nonzero(col)}/{len(col)}")
    print()
    for nk in f['nuisance_weights'].keys():
        nw = f['nuisance_weights'][nk][()]
        print(f"  {nk:40s}  mean={nw.mean():.4e}  nonzero={np.count_nonzero(nw)}/{len(nw)}")

Batch 1
Events: 249144
Observables: 3
Benchmark weights: (249144, 6)
Nuisance weight datasets: ['scale_corr_nuisance_param_0_down', 'scale_corr_nuisance_param_0_up', 'scale_muf_nuisance_param_0_down', 'scale_muf_nuisance_param_0_up', 'scale_mur_nuisance_param_0_down', 'scale_mur_nuisance_param_0_up']
Nuisance a/b shape: (3, 249144)
Morphing matrix: (6, 6)

  sm                              mean=4.5310e-09  nonzero=249144/249144
  w                               mean=1.8391e-07  nonzero=249144/249144
  neg_w                           mean=2.3944e-07  nonzero=249144/249144
  ww                              mean=2.2156e-07  nonzero=249144/249144
  neg_ww                          mean=2.2727e-07  nonzero=249144/249144
  morphing_basis_vector_5         mean=3.4086e-07  nonzero=249144/249144

  scale_corr_nuisance_param_0_down          mean=4.7434e-09  nonzero=249144/249144
  scale_corr_nuisance_param_0_up            mean=4.3366e-09  nonzero=249144/249144
  scale_muf_nuisance_param_0_down   